In [21]:
import cv2
import numpy as np
from pdf2image import convert_from_path
import os
import glob

# ====================== AYARLAR ======================
pdf_folder = 'pdfler'                    # Tüm PDF'lerin olduğu klasör
output_folder = 'optik_kirpilmis_resimler'
dpi = 300

# Kare algılama ayarları (önceki versiyondan optimize edilmiş hali)
BLUR_KERNEL = (5, 5)
MIN_SQUARE_AREA = 800
MAX_SQUARE_AREA = 40000
ASPECT_TOLERANCE = 0.25
SIDE_VARIANCE_MAX = 0.20
ADAPTIVE_BLOCK_SIZE = 21
ADAPTIVE_C = 5

DEBUG = False                            # True yaparsan her sayfa için DEBUG resmi kaydeder

# Klasörleri oluştur
os.makedirs(output_folder, exist_ok=True)
os.makedirs(pdf_folder, exist_ok=True)

# pdfler klasöründeki tüm .pdf dosyalarını bul
pdf_files = glob.glob(os.path.join(pdf_folder, '*.pdf'))

if not pdf_files:
    print(f"⚠️ '{pdf_folder}' klasöründe hiç .pdf dosyası bulunamadı!")
    print("Lütfen PDF dosyalarını 'pdfler' klasörüne koyun.")
    exit()

print(f"✅ {len(pdf_files)} adet PDF dosyası bulundu. İşlem başlıyor...\n")

def detect_squares(img):
    """Siyah kareleri tespit et (önceki versiyondan aynı)"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, BLUR_KERNEL, 0)

    squares = []
    methods = [
        cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              cv2.THRESH_BINARY_INV, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C),
        cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1],
        cv2.threshold(blurred, 70, 255, cv2.THRESH_BINARY_INV)[1]
    ]

    for thresh in methods:
        kernel = np.ones((3, 3), np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < MIN_SQUARE_AREA or area > MAX_SQUARE_AREA:
                continue

            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.018 * peri, True)
            if len(approx) != 4 or not cv2.isContourConvex(approx):
                continue

            x, y, w, h = cv2.boundingRect(approx)
            aspect = w / h
            if not (1 - ASPECT_TOLERANCE <= aspect <= 1 + ASPECT_TOLERANCE):
                continue

            pts = approx.reshape(4, 2).astype(np.float32)
            sides = [np.linalg.norm(pts[i] - pts[(i + 1) % 4]) for i in range(4)]
            if np.var(sides) > (np.mean(sides) * SIDE_VARIANCE_MAX) ** 2:
                continue

            squares.append((x, y, w, h, area))

    # Tekrar eden kareleri temizle
    unique_squares = []
    for s in squares:
        if not any(abs(s[0] - u[0]) < 15 and abs(s[1] - u[1]) < 15 for u in unique_squares):
            unique_squares.append(s)
    return unique_squares


# ====================== ANA İŞLEM DÖNGÜSÜ ======================
total_processed = 0

for pdf_path in pdf_files:
    pdf_name = os.path.basename(pdf_path).replace('.pdf', '')
    print(f"📄 İşleniyor: {pdf_name}.pdf")

    try:
        pages = convert_from_path(pdf_path, dpi=dpi)
        print(f"   → {len(pages)} sayfa tespit edildi.")
    except Exception as e:
        print(f"   ❌ PDF okunamadı: {e}")
        continue

    for page_idx, page in enumerate(pages):
        img = cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR)
        orig_h, orig_w = img.shape[:2]

        squares = detect_squares(img)

        if len(squares) < 4:
            print(f"   Sayfa {page_idx+1}: Yeterli kare yok → sabit crop uygulanıyor.")
            crop_w = int(orig_w * 0.50)
            crop_h = int(orig_h * 0.60)
            cropped = img[orig_h - crop_h : orig_h, orig_w - crop_w : orig_w]
        else:
            # Merkez noktaları ile sıralama
            markers = [(x + w//2, y + h//2, x, y, w, h, area) for x, y, w, h, area in squares]
            markers.sort(key=lambda m: m[1])  # Y eksenine göre üstten alta

            top = markers[:2]
            bottom = markers[-2:]

            tl = min(top, key=lambda m: m[0])
            tr = max(top, key=lambda m: m[0])
            bl = min(bottom, key=lambda m: m[0])
            br = max(bottom, key=lambda m: m[0])

            # ====================== İSTEDİĞİN KESİM MANTIĞI ======================
            crop_left   = max(tl[2] + tl[4], bl[2] + bl[4])   # Sol karelerin sağ kenarı
            crop_top    = max(tl[3] + tl[5], tr[3] + tr[5])   # Üst karelerin alt kenarı
            crop_right  = br[2] + br[4]                       # Sağ alt karenin sağ kenarı
            crop_bottom = min(bl[3], br[3])                   # Alt karelerin üst kenarı

            if crop_top >= crop_bottom or crop_left >= crop_right:
                print(f"   Sayfa {page_idx+1}: Koordinat hatası → sabit crop")
                cropped = img[orig_h - int(orig_h*0.60):orig_h, orig_w - int(orig_w*0.50):orig_w]
            else:
                cropped = img[crop_top:crop_bottom, crop_left:crop_right]
                print(f"   Sayfa {page_idx+1}: ✅ Başarılı kesim → {cropped.shape[1]}x{cropped.shape[0]}")

        # Dosya adı: pdf_adi_sayfa_numarasi.png
        file_name = f"{pdf_name}_sayfa_{page_idx+1}.png"
        save_path = os.path.join(output_folder, file_name)
        cv2.imwrite(save_path, cropped)
        total_processed += 1

        # DEBUG (isteğe bağlı)
        if DEBUG and len(squares) >= 4:
            debug_img = img.copy()
            for label, m in [("TL", tl), ("TR", tr), ("BL", bl), ("BR", br)]:
                x, y, w, h = m[2], m[3], m[4], m[5]
                cv2.rectangle(debug_img, (x, y), (x + w, y + h), (0, 0, 255), 8)
                cv2.putText(debug_img, label, (x + 20, y + 55), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 6)
            cv2.rectangle(debug_img, (crop_left, crop_top), (crop_right, crop_bottom), (0, 255, 0), 5)
            cv2.imwrite(os.path.join(output_folder, f"DEBUG_{pdf_name}_sayfa_{page_idx+1}.png"), debug_img)

    print(f"   → {pdf_name}.pdf tamamlandı.\n")

print(f"\n🎉 Tüm işlemler tamamlandı!")
print(f"   Toplam işlenen sayfa: {total_processed}")
print(f"   Çıktı klasörü: '{output_folder}'")

✅ 3 adet PDF dosyası bulundu. İşlem başlıyor...

📄 İşleniyor: sinav1.pdf
   → 36 sayfa tespit edildi.
   Sayfa 1: ✅ Başarılı kesim → 926x1447
   Sayfa 2: ✅ Başarılı kesim → 928x1447
   Sayfa 3: ✅ Başarılı kesim → 929x1447
   Sayfa 4: ✅ Başarılı kesim → 929x1446
   Sayfa 5: ✅ Başarılı kesim → 926x1446
   Sayfa 6: ✅ Başarılı kesim → 927x1447
   Sayfa 7: ✅ Başarılı kesim → 927x1446
   Sayfa 8: ✅ Başarılı kesim → 924x1446
   Sayfa 9: ✅ Başarılı kesim → 926x1447
   Sayfa 10: ✅ Başarılı kesim → 927x1446
   Sayfa 11: ✅ Başarılı kesim → 926x1447
   Sayfa 12: ✅ Başarılı kesim → 927x1447
   Sayfa 13: ✅ Başarılı kesim → 928x1447
   Sayfa 14: ✅ Başarılı kesim → 927x1447
   Sayfa 15: ✅ Başarılı kesim → 927x1446
   Sayfa 16: ✅ Başarılı kesim → 927x1445
   Sayfa 17: ✅ Başarılı kesim → 928x1446
   Sayfa 18: ✅ Başarılı kesim → 926x1446
   Sayfa 19: ✅ Başarılı kesim → 929x1446
   Sayfa 20: ✅ Başarılı kesim → 930x1447
   Sayfa 21: ✅ Başarılı kesim → 930x1446
   Sayfa 22: ✅ Başarılı kesim → 928x1446
   Sa

In [22]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob

# ====================== AYARLAR & CEVAP ANAHTARI ======================
folder_path = 'optik_kirpilmis_resimler'  # Klasör adı
threshold_percent = 25 
soru_puanı = 5

cevap_anahtari = [
    "D", "Y", "Y",              # 1a, 1b, 1c
    "A", "D", "C", "B", "B",    # 2 - 6
    "C", "A",                   # 7 - 8
    "E", "C", "B", "A", "E", "C", "A", "E", "C", "D" # 9 - 18
]

# ====================== MAPS ======================
num_columns_map = [
    ["1", "2", "3"], ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
    ["A", "B", "D", "F", "G", "İ", "O", "T"], ["H", "I", "N", "P", "R", "Y"],
    ["A", "G", "İ", "L", "Ş", "T", "Z"], ["A", "E", "F", "G", "İ", "Ö", "P", "R", "S", "T", "Z"],
    ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"], ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
    ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"], ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
]

answers_map_1 = [
    {"label": "1a", "opts": ["D", "Y"]}, {"label": "1b", "opts": ["D", "Y"]}, {"label": "1c", "opts": ["D", "Y"]},
    {"label": "2",  "opts": ["A", "B", "C", "D", "E"]}, {"label": "3",  "opts": ["A", "B", "C", "D", "E"]},
    {"label": "4",  "opts": ["A", "B", "C", "D", "E"]}, {"label": "5",  "opts": ["A", "B", "C", "D", "E"]},
    {"label": "6",  "opts": ["A", "B", "C", "D", "E"]}, {"label": "7",  "opts": ["A", "B", "C", "D", "E"]},
    {"label": "8",  "opts": ["A", "B", "C", "D", "E"]}
]
answers_map_2 = [{"label": str(i), "opts": ["A", "B", "C", "D", "E"]} for i in range(9, 19)]

# ====================== FONKSİYONLAR ======================

def get_roi_properly(img, title):
    cv2.namedWindow(title, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(title, 800, 1000)
    bbox = cv2.selectROI(title, img, fromCenter=False, showCrosshair=True)
    cv2.destroyWindow(title)
    return bbox

def analyze_area(image, box, row_count, col_count, mapping, mode="col"):
    x, y, w, h = box
    if w == 0 or h == 0: return None, []
    roi = image[y:y+h, x:x+w].copy()
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 160, 255, cv2.THRESH_BINARY_INV)
    
    detected = []
    r_h, c_w = h // row_count, w // col_count
    outer = col_count if mode=="col" else row_count
    inner = row_count if mode=="col" else col_count
    
    for i in range(outer):
        intensities = []
        for j in range(inner):
            curr_x, curr_y = (i*c_w, j*r_h) if mode=="col" else (j*c_w, i*r_h)
            cell = thresh[curr_y:curr_y+r_h, curr_x:curr_x+c_w]
            density = (cv2.countNonZero(cell) / cell.size) * 100
            intensities.append(density)
            cv2.rectangle(roi, (curr_x, curr_y), (curr_x+c_w, curr_y+r_h), (0, 255, 0), 1)

        max_idx = np.argmax(intensities)
        if intensities[max_idx] > threshold_percent:
            if mode == "col":
                curr_x, curr_y = i * c_w, max_idx * r_h
                val = mapping[i][max_idx] if max_idx < len(mapping[i]) else "?"
            else:
                curr_x, curr_y = max_idx * c_w, i * r_h
                val = mapping[i]["opts"][max_idx] if max_idx < len(mapping[i]["opts"]) else "Hata"
            
            cv2.rectangle(roi, (curr_x, curr_y), (curr_x+c_w, curr_y+r_h), (255, 0, 0), 2)
            detected.append(val)
        else:
            detected.append("Boş")
            
    return roi, detected

# ====================== TOPLU İŞLEME BAŞLIYOR ======================

# Klasördeki tüm PNG dosyalarını al
image_files = glob.glob(os.path.join(folder_path, "*.png"))

if not image_files:
    print(f"Hata: {folder_path} klasöründe PNG dosyası bulunamadı!")
    exit()

# 1. ADIM: KOORDİNATLARI BELİRLEME (Sadece ilk resim üzerinden)
print(f"Koordinat belirleme için {os.path.basename(image_files[0])} açılıyor...")
first_img = cv2.imread(image_files[0])

roi_num_box = get_roi_properly(first_img, "1. Ogrenci No (REFERANS)")
roi_ans1_box = get_roi_properly(first_img, "2. Sorular 1-8 (REFERANS)")
roi_ans2_box = get_roi_properly(first_img, "3. Sorular 9-18 (REFERANS)")

all_student_data = []
all_student_scores = []

# 2. ADIM: TÜM DOSYALARI DÖNGÜ İLE TARA
print("\nTarama başlatılıyor...")
for file in image_files:
    file_name = os.path.basename(file)
    img = cv2.imread(file)
    
    if img is None:
        print(f"Hata: {file_name} okunamadı, atlanıyor.")
        continue

    # Analizleri yap (Referans koordinatları kullanarak)
    _, num_res = analyze_area(img, roi_num_box, 11, 10, num_columns_map, mode="col")
    _, ans1_res = analyze_area(img, roi_ans1_box, 10, 5, answers_map_1, mode="row")
    _, ans2_res = analyze_area(img, roi_ans2_box, 10, 5, answers_map_2, mode="row")

    ogrenci_no = "".join(num_res).replace("Boş", "0")
    tum_cevaplar = ans1_res + ans2_res

    # Excel Sayfa 1 için veri (Şıklar)
    all_student_data.append([ogrenci_no] + tum_cevaplar)

    # Excel Sayfa 2 için veri (Puanlar)
    puanlar = [soru_puanı if tum_cevaplar[i] == cevap_anahtari[i] else 0 for i in range(len(tum_cevaplar))]
    toplam = sum(puanlar)
    all_student_scores.append([ogrenci_no] + puanlar + [toplam])
    
    print(f"İşlendi: {file_name} -> No: {ogrenci_no} | Puan: {toplam}")

# ====================== EXCEL KAYIT ======================
sutun_isimleri = ["Öğrenci No"] + [f"Soru_{m['label']}" for m in (answers_map_1 + answers_map_2)]
df1 = pd.DataFrame(all_student_data, columns=sutun_isimleri)

sutun_isimleri_puan = sutun_isimleri + ["Toplam Puan"]
df2 = pd.DataFrame(all_student_scores, columns=sutun_isimleri_puan)

with pd.ExcelWriter("Tum_Optik_Sonuclar.xlsx", engine="openpyxl") as writer:
    df1.to_excel(writer, sheet_name="Isaretlenenler", index=False)
    df2.to_excel(writer, sheet_name="Puanlar", index=False)

print("\n" + "="*40)
print(f"Toplam {len(all_student_data)} kağıt işlendi.")
print(f"Excel dosyası oluşturuldu: Tum_Optik_Sonuclar.xlsx")
print("="*40)

Koordinat belirleme için sinav1_sayfa_1.png açılıyor...

Tarama başlatılıyor...
İşlendi: sinav1_sayfa_1.png -> No: 25BILP0072 | Puan: 85
İşlendi: sinav1_sayfa_10.png -> No: 25BILP0064 | Puan: 50
İşlendi: sinav1_sayfa_11.png -> No: 25BILP0058 | Puan: 55
İşlendi: sinav1_sayfa_12.png -> No: 25BILP0078 | Puan: 50
İşlendi: sinav1_sayfa_13.png -> No: 25BILP0075 | Puan: 90
İşlendi: sinav1_sayfa_14.png -> No: 25BILP0021 | Puan: 65
İşlendi: sinav1_sayfa_15.png -> No: 25BILP0034 | Puan: 90
İşlendi: sinav1_sayfa_16.png -> No: 25BILP0027 | Puan: 40
İşlendi: sinav1_sayfa_17.png -> No: 23BILP0004 | Puan: 40
İşlendi: sinav1_sayfa_18.png -> No: 25BILP0017 | Puan: 50
İşlendi: sinav1_sayfa_19.png -> No: 25BILP0084 | Puan: 90
İşlendi: sinav1_sayfa_2.png -> No: 24BILP0033 | Puan: 40
İşlendi: sinav1_sayfa_20.png -> No: 25BILP0024 | Puan: 60
İşlendi: sinav1_sayfa_21.png -> No: 21BILP0023 | Puan: 75
İşlendi: sinav1_sayfa_22.png -> No: 25BILP0070 | Puan: 55
İşlendi: sinav1_sayfa_23.png -> No: 25BILP0037 | Pua

PermissionError: [Errno 13] Permission denied: 'Tum_Optik_Sonuclar.xlsx'

In [ ]:
import cv2
import os
import glob
import numpy as np

# ====================== AYARLAR ======================
input_folder = 'optik_kirpilmis_resimler'
output_folder = 'dogrulama_resimleri' # Kontrol edeceğin resimler burada toplanacak

# Eğer klasör yoksa oluştur
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print(f"Doğrulama resimleri '{output_folder}' klasörüne kaydedilecek...")

# ====================== İŞLEME DÖNGÜSÜ ======================

# Not: Bir önceki hücrede belirlediğin 'roi_num_box', 'roi_ans1_box' ve 
# 'roi_ans2_box' değişkenlerinin bellekte (RAM'de) hala duruyor olması gerekir.

image_files = glob.glob(os.path.join(input_folder, "*.png"))

for file in image_files:
    file_name = os.path.basename(file)
    img = cv2.imread(file)
    
    if img is None: continue

    # 1. Analizleri yap ve görselleştirilmiş (çizimli) hallerini al
    # analyze_area fonksiyonun roi'yi çizip geri döndürdüğü için onu kullanıyoruz
    roi_n, _ = analyze_area(img, roi_num_box, 11, 10, num_columns_map, mode="col")
    roi_a1, _ = analyze_area(img, roi_ans1_box, 10, 5, answers_map_1, mode="row")
    roi_a2, _ = analyze_area(img, roi_ans2_box, 10, 5, answers_map_2, mode="row")

    # 2. Bu üç parçayı tek bir resimde yan yana birleştir (Opsiyonel ama hızlı kontrol için en iyisi)
    # Resim yüksekliklerini eşitlemek için en büyük yüksekliği bulalım
    h1, w1 = roi_n.shape[:2]
    h2, w2 = roi_a1.shape[:2]
    h3, w3 = roi_a2.shape[:2]
    max_h = max(h1, h2, h3)

    # Resimleri aynı yüksekliğe getir (Altına siyah boşluk ekleyerek)
    def pad_image(image, target_h):
        h, w = image.shape[:2]
        pad = np.zeros((target_h - h, w, 3), dtype=np.uint8)
        return np.vstack((image, pad))

    roi_n_pad = pad_image(roi_n, max_h)
    roi_a1_pad = pad_image(roi_a1, max_h)
    roi_a2_pad = pad_image(roi_a2, max_h)

    # Aralara ince beyaz bir çizgi ekleyerek yan yana birleştir
    divider = np.ones((max_h, 10, 3), dtype=np.uint8) * 255
    combined_res = np.hstack((roi_n_pad, divider, roi_a1_pad, divider, roi_a2_pad))

    # 3. Kaydet
    save_path = os.path.join(output_folder, f"check_{file_name}")
    cv2.imwrite(save_path, combined_res)

print(f"İşlem bitti! Toplam {len(image_files)} adet doğrulama resmi hazır.")